# D3-03b Optional extension — Prospective AWARE 2.0

This optional notebook showcases the use of **“prospective AWARE 2.0”**.

Here, we reuse the Swiss **`Irrigating`** activity from the BAFU database. This keeps the focus on the question:

> How does the water-scarcity result change when the characterisation factors depend on both a future scenario and a year?

## Learning goals

By the end of this notebook, you should be able to:

- evaluate one fixed inventory with prospective AWARE 2.0 factors;
- compare deterministic results across SSP scenarios and years;
- explain how `edges` interpolates between source years;
- distinguish a deterministic country-average result from stochastic basin-level variability.

This is an **advanced, optional extension** to D3-03. It assumes that the `aalborg-rlcia-2026` project and the `bafu` database already exist.

## Background and provenance

The underlying AWARE 2.0 method is described in:

**Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A.-M.** (2025). *The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0.* Journal of Industrial Ecology. https://doi.org/10.1111/jiec.70023

The prospective factors are from:

**Seitfudem, G., Berger, M., & Boulay, A.-M.** (2026). *Harnessing model ensembles to assess uncertainty and provide prospective characterization factors for AWARE2.0.*

The compressed method file included with this course is a fallback copy of the data used by the original `edges` publication notebook.

## Workflow

1. Select the irrigation activity.
2. Inspect its direct withdrawal and return flows.
3. Load the prospective AWARE method.
4. Map the inventory once.
5. Re-evaluate the factors for several scenarios and years.
6. Sample basin-level variability for one scenario and year.

## 1) Import packages and check compatibility

In [ ]:
import gzip
import json
import shutil
import tempfile
from pathlib import Path

import bw2data as bd
import edges
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from edges import EdgeLCIA

## 2) Select and inspect the functional unit

We select by **name, reference product, and location**. Requiring exactly one match prevents us from silently calculating the wrong activity.

In [ ]:
bd.projects.set_current("aalborg-rlcia-2026")
bafu = bd.Database("bafu")

matches = [
    activity
    for activity in bafu
    if activity["name"] == "Irrigating"
    and activity["reference product"] == "Irrigating"
    and activity.get("location") == "CH"
]

if len(matches) != 1:
    raise ValueError(f"Expected one Swiss Irrigating activity; found {len(matches)}.")

irrigating = matches[0]
functional_unit = {irrigating: 1}

print(f"Functional unit: 1 {irrigating['unit']} of {irrigating['name']}")
print(f"Location: {irrigating['location']}")

### Direct water balance

The activity withdraws river water and returns most of it to the environment. AWARE assigns opposite-signed characterization factors to withdrawal and return flows.

The plot shows inventory amounts, not impact scores.

In [ ]:
water_balance = {}

for exchange in irrigating.biosphere():
    flow = exchange.input
    categories = tuple(flow.get("categories", ()))

    if flow["name"] == "Water, river" and categories == ("natural resource", "in water"):
        water_balance["River water withdrawn"] = float(exchange["amount"])
    elif flow["name"] == "Water" and categories == ("water",):
        water_balance["Water returned"] = float(exchange["amount"])

if set(water_balance) != {"River water withdrawn", "Water returned"}:
    raise ValueError("The expected withdrawal and return flows were not both found.")

fig, ax = plt.subplots(figsize=(7, 3.2))
labels = ["Water returned", "River water withdrawn"]
amounts = [water_balance[label] for label in labels]
bars = ax.barh(labels, amounts, color=["#72B7B2", "#4C78A8"])
ax.bar_label(bars, fmt="%.0f m³")
ax.set_xlabel("Direct inventory amount [m³ per hectare]")
ax.set_xlim(0, max(amounts) * 1.18)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

net_consumption = water_balance["River water withdrawn"] - water_balance["Water returned"]
print(f"Net direct water consumption: {net_consumption:.0f} m³ per hectare")

## 3) Load and inspect the prospective method

Recent `edges` installations package this method directly. The course also includes a compressed fallback so that the notebook remains self-contained.

The fallback is decompressed to the system temporary folder; the large JSON file is not duplicated inside the repository.

In [ ]:
method_filename = "AWARE 2.0 prospective_Country_all_yearly.json"
packaged_method_path = Path(edges.__file__).resolve().parent / "data" / method_filename

asset_candidates = [
    Path("assets/d3-03b/prospective_aware_2_country_all_yearly.json.gz"),
    Path("tutorials/DAY 3/assets/d3-03b/prospective_aware_2_country_all_yearly.json.gz"),
]

if packaged_method_path.exists():
    method_path = packaged_method_path
    method_source = "installed edges package"
else:
    asset_path = next((path for path in asset_candidates if path.exists()), None)
    if asset_path is None:
        raise FileNotFoundError("Could not find the packaged method or the course fallback asset.")

    method_path = Path(tempfile.gettempdir()) / method_filename
    with gzip.open(asset_path, "rb") as compressed, method_path.open("wb") as target:
        shutil.copyfileobj(compressed, target)
    method_source = "compressed course asset"

method_data = json.loads(method_path.read_text())
print(f"Loaded from: {method_source}")

The method contains:

- three SSP scenarios;
- seven years;
- country-average rules for deterministic calculations;
- basin-specific distributions for stochastic calculations.

Between source years, `edges` uses linear interpolation. Outside the source range, it uses the nearest endpoint.

In [ ]:
source_years = [int(year) for year in method_data["interpolation"]["source_years"]]
scenarios = list(method_data["parameters"])

method_overview = {
    "Method": method_data["name"],
    "Unit": method_data["unit"],
    "Scenarios": ", ".join(scenarios),
    "Source years": ", ".join(map(str, source_years)),
    "Characterization rules": f"{len(method_data['exchanges']):,}",
    "Uncertainty definitions": f"{len(method_data['uncertainties']):,}",
    "Interpolation": method_data["interpolation"]["method"],
}

for label, value in method_overview.items():
    print(f"{label}: {value}")

## 4) Map the inventory once

The technosphere inventory does not change in this exercise. We therefore calculate and spatially map it only once.

Later, `evaluate_cfs(...)` changes the scenario and year of the **characterization factors**, and `lcia()` recalculates the score.

In [ ]:
prospective_lca = EdgeLCIA(
    demand=functional_unit,
    filepath=str(method_path),
    use_distributions=False,
)

prospective_lca.lci()
prospective_lca.apply_strategies()

print("Inventory calculated and exchanges mapped.")

## 5) Compare scenarios and years

We evaluate seven years. Most fall between the method's source years, so this also exercises interpolation.

Before running the next cell, make a prediction:

> Will the three scenario lines be identical, converge, or diverge toward 2049?

In [ ]:
evaluation_years = [2020, 2025, 2030, 2035, 2040, 2045, 2050]
records = []

for scenario in scenarios:
    for year in evaluation_years:
        prospective_lca.evaluate_cfs(
            scenario=scenario,
            scenario_idx=year,
        )
        prospective_lca.lcia()

        records.append(
            {
                "scenario": scenario,
                "year": year,
                "score": float(prospective_lca.score),
            }
        )

deterministic_results = pd.DataFrame(records)

In [ ]:
scenario_colors = {
    "SSP126": "#2A9D8F",
    "SSP370": "#F4A261",
    "SSP585": "#C44E52",
}

fig, ax = plt.subplots(figsize=(8, 4.6))

for scenario, group in deterministic_results.groupby("scenario", sort=False):
    ax.plot(
        group["year"],
        group["score"],
        marker="o",
        linewidth=2,
        color=scenario_colors[scenario],
        label=scenario,
    )

ax.set(
    xlabel="Evaluation year",
    ylabel=f"AWARE score [{method_data['unit']} / ha irrigated]",
    title="Prospective water-scarcity impact of one hectare of irrigation",
)
ax.set_xticks(evaluation_years)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Scenario", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Put the change into words

The graph is the main result. The short summary below reports the start, end, and relative change without printing the full results table.

In [ ]:
for scenario in scenarios:
    group = deterministic_results.query("scenario == @scenario").set_index("year")
    start_score = group.loc[2020, "score"]
    end_score = group.loc[2050, "score"]
    relative_change = (end_score / start_score - 1) * 100

    print(
        f"{scenario}: {start_score:.1f} → {end_score:.1f} "
        f"{method_data['unit']} ({relative_change:+.0f}%)"
    )

## 6) Explore basin-level variability

The deterministic score uses country-average factors. The stochastic calculation instead samples the basin-specific CFs and their weights for one scenario/year.

We keep the inventory fixed, so this distribution represents **characterization-factor variability and uncertainty**, not inventory uncertainty.

The original publication example uses 1,000 iterations. We use 500 by default to keep this optional exercise responsive; increase the value for a more stable distribution.

In [ ]:
stochastic_scenario = "SSP585"
stochastic_year = 2035
iterations = 500

deterministic_lca = EdgeLCIA(
    demand=functional_unit,
    filepath=str(method_path),
    scenario=stochastic_scenario,
    #use_distributions=True,
    #iterations=iterations,
    #random_seed=42,
)

deterministic_lca.lci()
deterministic_lca.apply_strategies()
deterministic_lca.evaluate_cfs(
    scenario=stochastic_scenario,
    scenario_idx=stochastic_year,
)
deterministic_lca.lcia()
deterministic_reference = deterministic_lca.score

stochastic_lca = EdgeLCIA(
    demand=functional_unit,
    filepath=str(method_path),
    scenario=stochastic_scenario,
    use_distributions=True,
    iterations=iterations,
    random_seed=42,
)

stochastic_lca.lci()
stochastic_lca.apply_strategies()
stochastic_lca.evaluate_cfs(
    scenario=stochastic_scenario,
    scenario_idx=stochastic_year,
)
stochastic_lca.lcia()

stochastic_scores = np.asarray(stochastic_lca.score, dtype=float).reshape(-1)

In [ ]:
mean_score = stochastic_scores.mean()
p05, median_score, p95 = np.quantile(stochastic_scores, [0.05, 0.50, 0.95])

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.hist(
    stochastic_scores,
    bins=30,
    color="#4C78A8",
    edgecolor="white",
    alpha=0.9,
)
ax.axvspan(p05, p95, color="#F4A261", alpha=0.18, label="Central 90%")
ax.axvline(
    deterministic_reference,
    color="black",
    linestyle="--",
    linewidth=2,
    label="Deterministic country average",
)
ax.axvline(
    mean_score,
    color="#C44E52",
    linewidth=2,
    label="Stochastic mean",
)
ax.set(
    xlabel=f"AWARE score [{method_data['unit']} / ha irrigated]",
    ylabel="Number of iterations",
    title=f"Basin-level variability: {stochastic_scenario}, {stochastic_year}",
)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Deterministic country average: {deterministic_reference:.1f}")
print(f"Stochastic mean:             {mean_score:.1f}")
print(f"Stochastic median:           {median_score:.1f}")
print(f"Central 90% interval:        {p05:.1f} to {p95:.1f}")

### Interpretation checkpoint

Discuss with a neighbor:

1. Why is the deterministic country-average value not necessarily equal to the stochastic median?
2. Which assumption changed in this notebook: the inventory, the characterization factors, or both?
3. What extra information would you need before using these results in a decision?

<details>
<summary>Suggested interpretation</summary>

The country-average result and the stochastic distribution summarize spatial information differently. Sampling basin-specific values and weights need not produce a symmetric distribution, so its mean and median can differ from the deterministic aggregate.

Only the characterization factors change here. The 2025 BAFU inventory and its supply chain remain fixed, so this is not a complete prospective LCA. A decision study would also need future foreground and background inventories, model-fit checks, and sensitivity analysis.
</details>

## Recap

- `EdgeLCIA` mapped the fixed inventory once and then re-evaluated scenario/year-dependent CFs.
- Prospective AWARE results increased over the evaluated period, but the size of the increase differed by SSP scenario.
- `edges` interpolated the 2026 parameters between the 2024 and 2029 source years.
- The stochastic calculation exposed basin-level variability that a single country-average score hides.
- These results are prospective **LCIA factors applied to a current inventory**, not a complete future-system scenario.